# Setup ACDWM - Google Colab

**Propósito**: Configurar ambiente Colab para executar comparação GBML vs River vs ACDWM

**Autor**: Claude Code  
**Data**: 2025-01-07

---

## Estrutura Esperada no Google Drive:

```
MyDrive/
└── DSL-AG-hybrid/              # Pasta principal (seus arquivos)
    ├── baseline_acdwm.py
    ├── baseline_river.py
    ├── compare_gbml_vs_river.py
    ├── config_comparison.yaml
    ├── data_converters.py
    ├── shared_evaluation.py
    ├── gbml_evaluator.py
    ├── metrics.py
    └── ... (outros arquivos)
```

**IMPORTANTE**: Ajuste o caminho `DRIVE_PATH` na célula 2 para apontar para sua pasta!


---
## 1. Montar Google Drive


In [ ]:
from google.colab import drive
import os

# Monta Google Drive
drive.mount('/content/drive')

print("\n[OK] Google Drive montado em: /content/drive")

---
## 2. Configurar Caminhos


In [ ]:
import sys
from pathlib import Path

# ========================================
# AJUSTE ESTE CAMINHO PARA SUA PASTA!
# ========================================
DRIVE_PATH = '/content/drive/MyDrive/DSL-AG-hybrid'

# Verifica se pasta existe
if not os.path.exists(DRIVE_PATH):
    print(f"[X] ERRO: Pasta nao encontrada: {DRIVE_PATH}")
    print("\nAjuste a variavel DRIVE_PATH para o caminho correto!")
    print("\nPastas disponiveis em MyDrive:")
    !ls -la /content/drive/MyDrive/
else:
    print(f"[OK] Pasta encontrada: {DRIVE_PATH}")
    
    # Adiciona ao Python path
    if DRIVE_PATH not in sys.path:
        sys.path.insert(0, DRIVE_PATH)
        print(f"[OK] Adicionado ao sys.path")
    
    # Muda para diretorio de trabalho
    os.chdir(DRIVE_PATH)
    print(f"[OK] Working directory: {os.getcwd()}")
    
    # Lista arquivos
    print("\nArquivos na pasta:")
    !ls -lh *.py | head -20

---
## 3. Instalar Dependências


In [ ]:
print("Instalando dependencias...\n")
print("="*70)

# River (modelos online)
print("\n[1/5] Instalando River...")
!pip install -q river

# DEAP (Genetic Algorithm para GBML)
print("[2/5] Instalando DEAP...")
!pip install -q deap

# Imbalanced-learn (metricas)
print("[3/5] Instalando imbalanced-learn...")
!pip install -q imbalanced-learn

# CVXPY (requerido pelo ACDWM)
print("[4/5] Instalando cvxpy...")
!pip install -q cvxpy

# Outras dependencias
print("[5/5] Instalando PyYAML, pandas...")
!pip install -q pyyaml pandas

print("\n" + "="*70)
print("[OK] Todas as dependencias instaladas!\n")

---
## 4. Verificar Instalação


In [ ]:
print("Verificando versoes instaladas...\n")
print("="*70)

import importlib

packages = [
    'numpy',
    'scipy', 
    'sklearn',
    'river',
    'deap',
    'imblearn',
    'cvxpy',
    'yaml',
    'pandas'
]

for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, '__version__', 'N/A')
        print(f"[OK] {pkg_name:15s} : {version}")
    except ImportError as e:
        print(f"[X] {pkg_name:15s} : NAO INSTALADO - {e}")

print("\n" + "="*70)
print("[OK] Verificacao concluida!\n")

---
## 5. Clonar Repositório ACDWM


In [ ]:
import os

ACDWM_PATH = os.path.join(DRIVE_PATH, 'ACDWM')

# Verifica se ja existe
if os.path.exists(ACDWM_PATH):
    print(f"[OK] Repositorio ACDWM ja existe em: {ACDWM_PATH}")
    print("\nArquivos encontrados:")
    !ls -lh {ACDWM_PATH}/*.py
else:
    print(f"Clonando repositorio ACDWM...\n")
    !cd {DRIVE_PATH} && git clone https://github.com/jasonyanglu/ACDWM.git
    
    if os.path.exists(ACDWM_PATH):
        print(f"\n[OK] ACDWM clonado com sucesso em: {ACDWM_PATH}")
        print("\nArquivos:")
        !ls -lh {ACDWM_PATH}/*.py
    else:
        print(f"\n[X] ERRO: Nao foi possivel clonar ACDWM")

print("\n[OK] ACDWM pronto para uso!")

---
## 6. Testar Importação dos Módulos


In [ ]:
print("Testando importacao dos modulos...\n")
print("="*70)

import sys

# Adiciona DRIVE_PATH ao path se necessario
if DRIVE_PATH not in sys.path:
    sys.path.insert(0, DRIVE_PATH)

try:
    # Modulos principais
    print("[1/8] Importando data_converters...")
    from data_converters import river_to_numpy, convert_chunks_river_to_numpy
    
    print("[2/8] Importando shared_evaluation...")
    from shared_evaluation import ChunkEvaluator, calculate_shared_metrics
    
    print("[3/8] Importando baseline_river...")
    from baseline_river import RiverEvaluator
    
    print("[4/8] Importando baseline_acdwm...")
    from baseline_acdwm import ACDWMEvaluator, import_acdwm_modules
    
    print("[5/8] Importando gbml_evaluator...")
    from gbml_evaluator import GBMLEvaluator
    
    print("[6/8] Importando metrics...")
    from metrics import calculate_gmean_contextual
    
    print("[7/8] Importando compare_gbml_vs_river...")
    import compare_gbml_vs_river
    
    print("[8/8] Testando importacao ACDWM...")
    acdwm_modules = import_acdwm_modules(ACDWM_PATH)
    print(f"    - dwmil: {acdwm_modules['dwmil']}")
    print(f"    - chunk_size_select: {acdwm_modules['chunk_size_select']}")
    
    print("\n" + "="*70)
    print("[OK] TODOS OS MODULOS IMPORTADOS COM SUCESSO!\n")
    
except Exception as e:
    print(f"\n[X] ERRO na importacao: {e}")
    import traceback
    traceback.print_exc()
    print("\nVerifique se todos os arquivos estao na pasta correta!")

---
## 7. Teste Rápido: ACDWM com Dados Sintéticos


In [ ]:
print("Executando teste rapido do ACDWM...\n")
print("="*70)

import numpy as np
from baseline_acdwm import ACDWMEvaluator

# Gera dados sinteticos
def generate_synthetic_chunk(n_samples=100, n_features=4, seed=42):
    np.random.seed(seed)
    X_river = []
    y_river = []
    
    for _ in range(n_samples):
        x_dict = {f'x_{i}': np.random.randn() for i in range(n_features)}
        X_river.append(x_dict)
        y = 1 if np.random.rand() > 0.5 else 0
        y_river.append(y)
    
    return X_river, y_river

# Cria evaluator
print("\n[1/4] Criando ACDWMEvaluator...")
evaluator = ACDWMEvaluator(
    acdwm_path=ACDWM_PATH,
    classes=[0, 1],
    evaluation_mode='train-then-test',
    theta=0.01
)
print(f"    [OK] Evaluator criado: {evaluator.model_name}")

# Gera 2 chunks
print("\n[2/4] Gerando dados sinteticos...")
chunks = []
for i in range(2):
    X_train, y_train = generate_synthetic_chunk(n_samples=150, seed=100+i)
    X_test, y_test = generate_synthetic_chunk(n_samples=50, seed=200+i)
    chunks.append((X_train, y_train, X_test, y_test))
print(f"    [OK] 2 chunks gerados (150 treino, 50 teste cada)")

# Executa avaliacao
print("\n[3/4] Executando avaliacao train-then-test...")
results = evaluator.evaluate_train_then_test(chunks)
print(f"    [OK] Avaliacao concluida")

# Exibe resultados
print("\n[4/4] Resultados:")
print("\n" + "="*70)
for i, result in enumerate(results):
    print(f"\nChunk {i+1}:")
    print(f"  Accuracy:      {result.get('accuracy', 0):.4f}")
    print(f"  G-mean:        {result.get('gmean', 0):.4f}")
    print(f"  F1-weighted:   {result.get('f1_weighted', 0):.4f}")
    print(f"  Ensemble size: {result['train_info']['ensemble_size']}")

print("\n" + "="*70)
print("[OK] TESTE RAPIDO CONCLUIDO COM SUCESSO!\n")
print("O ACDWM esta funcionando corretamente no Colab!")

---
## 8. Verificar Configuração (config_comparison.yaml)


In [ ]:
import yaml
import os

config_file = os.path.join(DRIVE_PATH, 'config_comparison.yaml')

if os.path.exists(config_file):
    print(f"[OK] Arquivo de configuracao encontrado: {config_file}\n")
    
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    
    print("Datasets configurados:")
    print("="*70)
    for dataset_name in config.get('datasets', {}).keys():
        print(f"  - {dataset_name}")
    
    print("\nParametros GA (GBML):")
    print("="*70)
    ga_params = config.get('ga_params', {})
    for key, value in ga_params.items():
        print(f"  {key:20s}: {value}")
    
    print("\n[OK] Configuracao validada!")
else:
    print(f"[X] ERRO: config_comparison.yaml nao encontrado em {DRIVE_PATH}")
    print("\nCertifique-se de que o arquivo esta na pasta correta!")

---
## 9. Resumo do Setup


In [ ]:
import os

print("\n" + "="*70)
print("RESUMO DO SETUP")
print("="*70)

print(f"\n[OK] Google Drive montado")
print(f"[OK] Working directory: {os.getcwd()}")
print(f"[OK] ACDWM path: {ACDWM_PATH}")

print("\n[OK] Dependencias instaladas:")
print("    - River (modelos online)")
print("    - DEAP (algoritmo genetico)")
print("    - imbalanced-learn (metricas)")
print("    - cvxpy (ACDWM)")
print("    - PyYAML, pandas")

print("\n[OK] Modulos importados:")
print("    - data_converters")
print("    - shared_evaluation")
print("    - baseline_river")
print("    - baseline_acdwm")
print("    - gbml_evaluator")
print("    - compare_gbml_vs_river")

print("\n[OK] ACDWM testado com sucesso!")

print("\n" + "="*70)
print("AMBIENTE CONFIGURADO E PRONTO PARA USO!")
print("="*70)

print("\nProximos passos:")
print("  1. Executar comparacao em 1 dataset (teste)")
print("  2. Executar comparacao completa (3 datasets)")
print("  3. Analisar resultados")

---
## Próximos Passos

Agora que o ambiente está configurado, você pode:

### Opção 1: Teste Rápido (1 dataset, poucos chunks)
```python
!cd {DRIVE_PATH} && python compare_gbml_vs_river.py \
    --stream RBF_Abrupt_Severe \
    --config config_comparison.yaml \
    --models HAT ARF \
    --chunks 2 \
    --chunk-size 1000 \
    --output test_results
```

### Opção 2: Experimento Completo
```python
!cd {DRIVE_PATH} && python run_comparison_colab.py
```

### Opção 3: Adicionar ACDWM à Comparação
- Primeiro, precisamos integrar ACDWM ao `compare_gbml_vs_river.py`
- Depois, executar comparação GBML vs River vs ACDWM
